# Lab 23-02 · SRAM Assists, Variation and Vmin

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize, stats
import sramtech as st

VDD, KPG, KPU, KPD, VT = 0.7, 100e-6, 80e-6, 100e-6, 0.3
VM = st.switching_threshold(VDD, KPD, KPU, VT, VT)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Assists in ngspice

*Predict* each value from §3.4 and §3.5 first.

In [2]:
pred = {"vm": VM,
        "vw0": st.write_node(VDD, KPG, KPU, VT, VT),
        "vw1": st.write_node(VDD, KPG, KPU, 0.35, 0.25),
        "vw2": st.write_node(VDD, KPG, KPU, 0.35, 0.25, -0.043),
        "vr0": st.read_node(VDD, KPG, KPD, VT, VT, 0.7),
        "vr1": st.read_node(VDD, KPG, KPD, VT, VT, 0.6)}
sim = ngspice("assist.cir")
for kk, vv in pred.items():
    print(f"{kk}: model {vv:.4f} V, ngspice {sim[kk]:.4f} V")

vm: model 0.3472 V, ngspice 0.3472 V
vw0: model 0.2211 V, ngspice 0.2211 V
vw1: model 0.4722 V, ngspice 0.4722 V
vw2: model 0.3472 V, ngspice 0.3472 V
vr0: model 0.1172 V, ngspice 0.1172 V
vr1: model 0.0716 V, ngspice 0.0716 V


## Part 2 · Write failures

*Predict*: with σ = 25 mV and no assist, is the failure rate closer to 10⁻¹ or 10⁻⁶?

In [3]:
SIG = 0.025


def failing_mismatch(v_bl):
    return optimize.brentq(lambda d: st.write_node(VDD, KPG, KPU, VT + d, VT - d, v_bl) - VM, 0, 0.29)


rng = np.random.default_rng(2302)
deltas = rng.normal(0, SIG, 20_000)
for v_bl in (0.0, -0.05, -0.10):
    ds = failing_mismatch(v_bl)
    mc = np.mean(deltas > ds)
    print(f"bitline {v_bl*1e3:5.0f} mV: failing mismatch {ds*1e3:.1f} mV, Q = {stats.norm.sf(ds/SIG):.3g}, Monte Carlo {mc:.3g}")
print(f"negative bitline for a 150 mV mismatch: {st.nbl_needed(VDD, KPG, KPU, 0.45, 0.15, VM)*1e3:.0f} mV")

bitline     0 mV: failing mismatch 24.6 mV, Q = 0.162, Monte Carlo 0.158
bitline   -50 mV: failing mismatch 54.1 mV, Q = 0.0153, Monte Carlo 0.0159
bitline  -100 mV: failing mismatch 83.6 mV, Q = 0.000414, Monte Carlo 0.00025
negative bitline for a 150 mV mismatch: -209 mV


## Part 3 · Vmin against array size

*Predict*: how much does 1,024 times more bits cost?

In [4]:
bits = np.logspace(10, 30, 41, base=2)
plt.figure()
for sig in (25.0, 20.4):
    plt.semilogx(bits, [st.vmin(st.sigma_needed(n), sig) for n in bits], label=f"sigma_VT = {sig} mV")
plt.xlabel("array size (bits)"); plt.ylabel("Vmin (V)"); plt.legend(); plt.show()
print(f"1 Mib -> 1 Gib at 25 mV: {st.vmin(st.sigma_needed(2**30), 25)-st.vmin(st.sigma_needed(2**20), 25):.3f} V")

1 Mib -> 1 Gib at 25 mV: 0.082 V


## Part 4 · Bitline length

*Predict* where the density gain flattens.

In [5]:
n = np.array([64, 128, 256, 512, 1024])
dens = 1 / np.array([st.area_per_bit(0.021, 1.0, int(k), 0.0012) for k in n])
tbl = np.array([st.bitline_time(int(k), 0.1e-15, 5e-15, 0.1, 20e-6) for k in n])
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].semilogx(n, dens, "o-"); ax[0].set_ylabel("Mb/mm^2")
ax[1].semilogx(n, tbl * 1e12, "o-"); ax[1].set_ylabel("bitline time (ps)"); ax[1].set_xlabel("cells per bitline")
plt.show()
for k, d_, t_ in zip(n, dens, tbl):
    print(f"{k:5d} cells: {d_:5.1f} Mb/mm^2, {t_*1e12:5.0f} ps")

   64 cells:  26.4 Mb/mm^2,    57 ps
  128 cells:  33.3 Mb/mm^2,    89 ps
  256 cells:  38.3 Mb/mm^2,   153 ps
  512 cells:  41.4 Mb/mm^2,   281 ps
 1024 cells:  43.1 Mb/mm^2,   537 ps
